# Proyecto Final Machine Learning
## Fase 1: Preprocesamiento y Visualización

**Dataset:** UCI HAR — Human Activity Recognition Using Smartphones  
**Equipo:** Grupo XX  
**Integrantes:** [Nombre 1] · [Nombre 2] · [Nombre 3]  
**Fecha:** [Fecha de entrega]

---

> **Objetivo:** Comprender el dataset HAR, preparar los datos para el modelado y comunicar los hallazgos exploratorios mediante visualizaciones.

## Configuración del entorno

Ejecuta esta celda primero para instalar las dependencias necesarias.

In [1]:
# Instalación de dependencias (ejecutar solo si es necesario)
# !pip install pandas numpy matplotlib seaborn scikit-learn ucimlrepo --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

# Semilla global para reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


---
## 1. Descarga y Carga del Dataset

El dataset UCI HAR está disponible en el UCI Machine Learning Repository. Contiene datos de 30 voluntarios realizando 6 actividades cotidianas con un smartphone en la cintura. Las señales del acelerómetro y giroscopio fueron procesadas para extraer 561 features.

In [2]:
import urllib.request
import zipfile
import os

# Descarga automática del dataset
URL = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'
ZIP_PATH = 'har_dataset.zip'
DATA_DIR = 'UCI HAR Dataset'

if not os.path.exists(DATA_DIR):
    print('Descargando dataset...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print('Descomprimiendo...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('.')
    os.remove(ZIP_PATH)
    print('Dataset listo.')
else:
    print('Dataset ya descargado.')

Dataset ya descargado.


### Código para el TODO 1: Cargar archivos del dataset

Este bloque lee los archivos de texto usando rutas relativas, extrae los nombres de las columnas y arma los DataFrames de entrenamiento y prueba.

In [ ]:
# Aseguramos que DATA_PATH esté definido (en la descarga se usó DATA_DIR)
DATA_PATH = 'UCI HAR Dataset'

# 1. Cargar nombres de features originales
features = pd.read_csv(f'{DATA_PATH}/features.txt', sep=r'\s+', header=None, names=['idx', 'feature'])
feature_names = features['feature'].tolist()

# 2. Desduplicar nombres de features añadiendo sufijos numéricos
seen = {}
unique_names = []
for name in feature_names:
    if name in seen:
        seen[name] += 1
        unique_names.append(f'{name}_{seen[name]}')
    else:
        seen[name] = 0
        unique_names.append(name)
feature_names = unique_names

# 3. ¡Ahora sí! Cargar el set completo de entrenamiento y prueba sin errores
X_train = pd.read_csv(f'{DATA_PATH}/train/X_train.txt', sep=r'\s+', header=None, names=feature_names)
y_train = pd.read_csv(f'{DATA_PATH}/train/y_train.txt', sep=r'\s+', header=None, names=['Activity_ID'])

X_test = pd.read_csv(f'{DATA_PATH}/test/X_test.txt', sep=r'\s+', header=None, names=feature_names)
y_test = pd.read_csv(f'{DATA_PATH}/test/y_test.txt', sep=r'\s+', header=None, names=['Activity_ID'])

print("=" * 50)
print("¡Todos los archivos del dataset se cargaron exitosamente!")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")
print("=" * 50)


### Código para el TODO 2: Mapear etiquetas numéricas

Transformamos los IDs numéricos (1 a 6) en nombres legibles de actividades para que los gráficos y reportes posteriores sean intuitivos.

In [ ]:
# TODO 2: Mapear etiquetas numéricas a nombres de actividad
ACTIVITY_LABELS = {
    1: 'WALKING',
    2: 'WALKING_UPSTAIRS',
    3: 'WALKING_DOWNSTAIRS',
    4: 'SITTING',
    5: 'STANDING',
    6: 'LAYING'
}

# Aplicar el mapeo a y_train e y_test
y_train_labels = y_train['Activity_ID'].map(ACTIVITY_LABELS)
y_test_labels  = y_test['Activity_ID'].map(ACTIVITY_LABELS)

print('Clases únicas:', sorted(y_train_labels.unique()))

---
## 2. Inspección Inicial del Dataset

Antes de cualquier procesamiento debemos conocer la estructura básica del dataset: dimensiones, tipos de datos, nulos y duplicados.

### Código para los TODOs 3, 4, 5, 6 y 7

Ejecuta consecutivamente estas celdas para extraer la radiografía estructural de tus datos.

In [ ]:
# TODO 3: Mostrar dimensiones de train y test
print('Dimensiones X_train:', X_train.shape)
print('Dimensiones X_test: ', X_test.shape)

In [ ]:
# TODO 4: Mostrar los primeros 5 registros y los tipos de datos
print("--- Primeros 5 registros de X_train ---")
display(X_train.head())

print("\n--- Conteo de tipos de datos en las columnas ---")
print(X_train.dtypes.value_counts())

In [ ]:
# TODO 5: Verificar valores faltantes (NaN)
missing_train = X_train.isnull().sum().sum()
missing_test  = X_test.isnull().sum().sum()
print(f'Valores faltantes en train: {missing_train}')
print(f'Valores faltantes en test:  {missing_test}')

In [ ]:
# TODO 6: Verificar duplicados en el set de entrenamiento
duplicates = X_train.duplicated().sum()
print(f'Filas duplicadas en X_train: {duplicates}')

In [ ]:
# TODO 7: Estadísticos descriptivos básicos
X_train.describe()

### 📝 Análisis en Markdown para la Inspección Inicial

**Observaciones del rango de valores:** Al analizar la salida de `df.describe()`, se observa que el valor mínimo de todas las variables está muy cercano o es exactamente **-1**, y el valor máximo es **1**.

**¿Necesitan normalizar? ¿Por qué?:** No, no es necesario aplicar una etapa adicional de normalización o estandarización (como `StandardScaler` o `MinMaxScaler`). Las 561 variables numéricas del dataset original ya han sido preprocesadas, normalizadas y acotadas en un rango estrictamente acotado entre **-1 y 1** por los autores del benchmark. Aplicar otra escala alteraría la estructura matemática balanceada que ya poseen estas señales inerciales.

---
## 3. Análisis de Balance de Clases

El desbalance de clases puede sesgar el modelo. Antes de modelar, examinamos la distribución de actividades.

### Código para los TODOs 8 y 9

In [ ]:
# TODO 8: Contar muestras por clase en el set de entrenamiento
class_counts = y_train_labels.value_counts()
print(class_counts)

In [ ]:
# TODO 9: Graficar distribución de clases con conteo y porcentaje
fig, ax = plt.subplots(figsize=(10, 5))

total_samples = len(y_train_labels)
# Crear gráfico de barras utilizando Seaborn
sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax, palette='viridis')

# Añadir etiquetas con conteo y porcentaje exacto sobre cada barra
for i, count in enumerate(class_counts.values):
    percentage = (count / total_samples) * 100
    ax.text(i, count + 15, f'{count}\n({percentage:.1f}%)', ha='center', va='bottom', fontsize=10)

ax.set_title('Distribución de Actividades — Set de Entrenamiento', fontsize=13)
ax.set_xlabel('Actividad')
ax.set_ylabel('Número de muestras')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 📝 Análisis en Markdown para el Balance de Clases

**¿Está balanceado el dataset?:** Sí, el dataset se encuentra bastante bien balanceado. Al observar los porcentajes, cada una de las 6 actividades cotidianas representa aproximadamente entre el **14% y el 19%** del total de las muestras de entrenamiento. No existe una clase ultra-mayoritaria que opaque por completo a las demás.

**¿Cómo podría afectar el desbalance al modelo y qué métricas usar?:** Si existiera un desbalance severo, el modelo tendería a aprender e inclinarse por predecir siempre la clase mayoritaria para asegurar un porcentaje alto de aciertos algorítmicos directos, ignorando las clases minoritarias. Aunque aquí el desbalance es mínimo, la métrica **Accuracy** simple puede ser engañosa si el modelo comete errores sistemáticos en una sola clase crítica. Por ello, para evaluar de forma robusta y justa, las métricas más adecuadas son el **F1-Score Macro** (que calcula el rendimiento por clase de forma independiente y saca un promedio no ponderado) y el **F1-Score Weighted** (que pondera según el volumen real de la clase).

---
## 4. Visualización y Exploración de Features

Con 561 features no podemos visualizarlas todas. Usamos técnicas inteligentes para identificar patrones y features relevantes.

### 4.1 Distribución de features clave (Boxplots)

Seleccionamos features representativas de los cuatro grupos: dominio tiempo/frecuencia × fuente cuerpo/gravedad.

### Código para los TODOs 10 y 11 (Boxplots)

In [ ]:
# TODO 10: Seleccionar al menos 4 features representativas (una por grupo)
# El dataset agrupa señales temporales (t) y de frecuencia (f), discriminando cuerpo (Body) y gravedad (Gravity)
SELECTED_FEATURES = [
    'tBodyAcc-mean()-X',     # Dominio Tiempo - Cuerpo
    'tGravityAcc-mean()-X',  # Dominio Tiempo - Gravedad
    'fBodyAcc-mean()-X',     # Dominio Frecuencia - Cuerpo
    'fBodyGyro-mean()-X'     # Dominio Frecuencia - Giroscopio (Cuerpo)
]

# Crear DataFrame auxiliar para graficar
train_plot = X_train[SELECTED_FEATURES].copy()
train_plot['Activity'] = y_train_labels.values

In [ ]:
# TODO 11: Graficar boxplots para cada feature seleccionada, coloreando por actividad
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(SELECTED_FEATURES):
    sns.boxplot(x='Activity', y=feat, data=train_plot, ax=axes[i], palette='Set2')
    axes[i].set_title(feat, fontsize=12)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_xlabel('')

plt.suptitle('Distribución de Features Clave por Actividad', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 📝 Análisis en Markdown para Boxplots

**¿Qué features separan mejor las clases?:** Las características ligadas a la gravedad, como `tGravityAcc-mean()-X`, muestran una capacidad de separación asombrosa entre dos macro-grupos: actividades **estáticas** (SITTING, STANDING, LAYING) y **dinámicas** (WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS). Esto ocurre porque la aceleración de la gravedad registra componentes muy estables y distintos dependiendo de la inclinación del teléfono en la cintura del usuario en reposo.

**¿Cuáles son menos informativas?:** Algunas variables de aceleración del cuerpo en el dominio del tiempo o de la frecuencia presentan rangos de distribución muy solapados entre actividades similares (por ejemplo, entre SITTING y STANDING), lo que significa que de forma aislada aportan poca información discriminatoria para separar esas clases específicas.

### 4.2 Mapa de calor de correlación

### Código para los TODOs 12 y 13 (Heatmap)

In [ ]:
# TODO 12: Seleccionar las 20 features con mayor varianza
top_var_features = X_train.var().nlargest(20).index.tolist()

# TODO 13: Calcular y graficar el heatmap de correlación
corr_matrix = X_train[top_var_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    ax=ax
)
ax.set_title('Mapa de Calor — Correlación entre Top-20 Features por Varianza', fontsize=13)
plt.tight_layout()
plt.show()

### 📝 Análisis en Markdown para Heatmap

**Implicaciones de la alta correlación:** Una alta correlación positiva o negativa entre variables indica la presencia de **multicolinealidad** (redundancia severa de información). Para modelos lineales como la Regresión Logística, esto puede desestabilizar la estimación de los coeficientes óptimos y perjudicar la convergencia. Para algoritmos basados en distancias como K-Nearest Neighbors (KNN), puede sobredimensionar la importancia de un eje sensor redundante. En contraste, los modelos basados en árboles de decisión o ensamblados (Random Forest) suelen lidiar bastante bien con esto, aunque la importancia individual de las features tienda a dispersarse.

### 4.3 Reducción de dimensionalidad — PCA

### Código para los TODOs 14 y 15 (PCA)

In [ ]:
# TODO 14: Aplicar PCA para reducir a 2 dimensiones
pca = PCA(n_components=2, random_state=RANDOM_STATE)

# Ajustar y transformar X_train
X_pca = pca.fit_transform(X_train)

df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['Activity'] = y_train_labels.values

In [ ]:
# TODO 15: Graficar espacio PCA coloreando por clase de actividad
fig, ax = plt.subplots(figsize=(10, 7))

sns.scatterplot(x='PC1', y='PC2', hue='Activity', data=df_pca, palette='tab10', alpha=0.7, ax=ax)

ax.set_title('Espacio PCA (2 componentes) — Coloreado por Actividad', fontsize=13)
ax.set_xlabel('Componente Principal 1')
ax.set_ylabel('Componente Principal 2')
ax.legend(title='Actividad', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Varianza explicada
print(f'Varianza explicada PC1: {pca.explained_variance_ratio_[0]:.3f}')
print(f'Varianza explicada PC2: {pca.explained_variance_ratio_[1]:.3f}')
print(f'Varianza acumulada total: {pca.explained_variance_ratio_.sum():.3f}')

### 📝 Análisis en Markdown para PCA

**Interpretación del espacio PCA:** La proyección en 2 dimensiones ratifica visualmente la existencia de los dos grandes patrones de movimiento. A la izquierda del gráfico, las actividades **dinámicas** (los tres tipos de caminata) forman un bloque claramente identificable. A la derecha, las **estáticas** se agrupan firmemente.

**¿Cuáles se solapan y qué sugiere?:** Existe un solapamiento muy evidente entre las clases **SITTING** (sentado) y **STANDING** (de pie). Físicamente tiene sentido, ya que el torso mantiene una verticalidad similar en reposo en ambos escenarios. Esto nos sugiere que, aunque el problema tiene una base de separabilidad lineal fuerte para discriminar el movimiento elemental, necesitaremos modelos con fronteras de decisión más complejas o hiperplanos no lineales (como un Support Vector Machine con Kernel RBF) para destrabar con precisión la confusión entre estar sentado o de pie.

### 4.4 Visualización adicional (libre)

Para cumplir con la quinta visualización obligatoria de la rúbrica, graficaremos la distribución de densidad estimada (KDE) de la magnitud del promedio de aceleración del cuerpo. Esto ilustra de manera elegante los niveles de energía cinética por actividad.

### Código para el TODO 16 (Visualización adicional libre)

In [ ]:
# TODO 16: Visualización adicional a elección del grupo (Gráfico KDE de Energía)
# Usaremos una variable clásica de energía o magnitud si está disponible, o una desviación estándar del movimiento temporal.
feat_adicional = 'tBodyAcc-std()-X'

fig, ax = plt.subplots(figsize=(10, 6))
for activity in y_train_labels.unique():
    subset = X_train[y_train_labels == activity]
    sns.kdeplot(subset[feat_adicional], label=activity, shade=True, alpha=0.25, ax=ax)

ax.set_title(f'Distribución de Densidad (KDE) de {feat_adicional} por Actividad', fontsize=13)
ax.set_xlabel('Desviación Estándar de Aceleración del Cuerpo (Eje X)')
ax.set_ylabel('Densidad de Probabilidad')
ax.legend(title='Actividad')
plt.tight_layout()
plt.show()

### 📝 Análisis en Markdown para la Visualización Adicional

**Análisis e interpretación:** Este gráfico de densidad (KDE) demuestra empíricamente que la variabilidad de la aceleración del cuerpo (`tBodyAcc-std()-X`) es un indicador directo de la intensidad física de la actividad. Las actividades **estáticas** se concentran agudamente en valores extremadamente bajos (cercanos a -1), reflejando ausencia de aceleración propia del movimiento. En cambio, las **caminatas** se desplazan hacia la derecha, destacando **WALKING_DOWNSTAIRS** (bajar escaleras) como una de las actividades con mayor dispersión de impactos cinéticos debido a la fuerza de gravedad descendente involucrada.

---
## 5. Preparación Final de Datos y Resumen

### Código para los TODOs 17 y 18

In [ ]:
# TODO 17: Confirmar que el split train/test viene predefinido (NO crear uno propio)
print('Estructura de splits verificada.')
print(f'Muestras de entrenamiento asignadas de forma fija: {X_train.shape[0]}')
print(f'Muestras de prueba asignadas de forma fija:        {X_test.shape[0]}')

### 📝 Análisis en Markdown sobre la importancia del Split Fijo

**¿Por qué es importante respetar el split original?:** En el dataset UCI HAR, las mediciones provienen de ventanas de tiempo continuas ejecutadas secuencialmente por 30 sujetos distintos. Los creadores separaron los datos **a nivel de sujeto** (por ejemplo, ciertos voluntarios van exclusivamente a entrenamiento y otros a prueba).

Si hiciéramos un split aleatorio tradicional (`train_test_split` mezclando todas las filas), estaríamos cometiendo **Data Leakage** (fuga de datos). Ventanas temporales consecutivas y altamente correlacionadas del mismo sujeto quedarían repartidas simultáneamente en ambos conjuntos. El modelo memorizaría los patrones particulares de movimiento de esos usuarios específicos, dándonos una precisión falsamente optimista en el set de prueba que caería estrepitosamente al intentar generalizar el sistema con personas completamente nuevas en el mundo real.

In [ ]:
# TODO 18: Exportar variables limpias para la Fase 2
# (Confirmar estados finales e imprimir celda de validación requerida)
print('=' * 50)
print('RESUMEN FINAL FASE 1')
print('=' * 50)
print(f'Dimensiones finales X_train: {X_train.shape}')
print(f'Dimensiones finales X_test:  {X_test.shape}')
print(f'Número de clases:            {len(ACTIVITY_LABELS)}')
print(f'Nombres de clases:           {list(ACTIVITY_LABELS.values())}')
print(f'Valores faltantes totales:   {X_train.isnull().sum().sum() + X_test.isnull().sum().sum()}')
print(f'Duplicados en train:         {X_train.duplicated().sum()}')
print('=' * 50)